In [0]:
# %sql
# DROP TABLE IF EXISTS Weather_Analytics.silver.silver_weather_clean;
# DROP TABLE IF EXISTS Weather_Analytics.silver.silver_weather_quarantine;
# DELETE FROM Weather_Analytics.silver.pipeline_control;

num_affected_rows
1


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.sql.window import Window

In [0]:
#audit_log
control_df = spark.sql("""
SELECT COALESCE(MAX(last_processed_version), -1) as last_version
FROM Weather_Analytics.silver.pipeline_control
""")

last_version = control_df.collect()[0]["last_version"]
control_df.display()


last_version
-1


In [0]:
latest_table_version = spark.sql("""
DESCRIBE HISTORY Weather_Analytics.bronze.bronze_weather
""").selectExpr("max(version)").collect()[0][0]

start_version = last_version + 1

if start_version > latest_table_version:
    print("No new data")
    cdf_df = spark.read.table("Weather_Analytics.bronze.bronze_weather").limit(0)
     
else:
    cdf_df = spark.read.format("delta") \
        .option("readChangeFeed", "true") \
        .option("startingVersion", start_version) \
        .table("Weather_Analytics.bronze.bronze_weather") \
        .filter(F.col("_change_type").isin("insert", "update_postimage"))
    print("Cdf is updated")

Cdf is updated


In [0]:
# JSON schema for parsing the sensor_payload column
schema = StructType([
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("pressure", DoubleType(), True),
    StructField("weather_condition", StringType(), True),
    StructField("rainfall", DoubleType(), True),
    StructField("uv_index", DoubleType(), True),
    StructField("visibility", DoubleType(), True)
])

In [0]:
if "sensor_payload" not in cdf_df.columns:
    cdf_df = cdf_df.withColumn("sensor_payload", F.lit(None))

cdf_df = cdf_df.withColumn(
    "parsed_json",
    F.from_json(F.col("sensor_payload").cast("string"), schema)
)

cdf_df = cdf_df.withColumn("temperature",F.coalesce(F.expr("try_cast(temperature as double)"),F.col("parsed_json.temperature"))) \
    .withColumn("humidity",F.coalesce(F.expr("try_cast(humidity as double)"),F.col("parsed_json.humidity"))) \
    .withColumn("wind_speed",F.coalesce(F.expr("try_cast(wind_speed as double)"),F.col("parsed_json.wind_speed"))) \
    .withColumn("pressure",F.coalesce(F.expr("try_cast(pressure as double)"),F.col("parsed_json.pressure"))) \
    .withColumn("rainfall",F.coalesce(F.expr("try_cast(rainfall as double)"),F.col("parsed_json.rainfall"))) \
    .withColumn("weather_condition",F.coalesce(F.col("weather_condition"),F.col("parsed_json.weather_condition"))) \
    .withColumn("uv_index",F.coalesce(F.expr("try_cast(uv_index as double)"),F.col("parsed_json.uv_index"))) \
    .withColumn("visibility",F.coalesce(F.expr("try_cast(visibility as double)"),F.col("parsed_json.visibility")))

cdf_df = cdf_df.drop("parsed_json", "sensor_payload")

In [0]:
from pyspark.sql import functions as f

df = cdf_df \
    .withColumn(
        "station_id",
        f.when((f.col("city") == "Chennai")  & (f.col("station_id") == "101"), "STN_CHN_01")
        .when((f.col("city") == "Chennai")   & (f.col("station_id") == "102"), "STN_CHN_02")
        .when((f.col("city") == "Bangalore") & (f.col("station_id") == "201"), "STN_BLR_01")
        .when((f.col("city") == "Bangalore") & (f.col("station_id") == "202"), "STN_BLR_02")
        .when((f.col("city") == "Hyderabad") & (f.col("station_id") == "301"), "STN_HYD_01")
        .when((f.col("city") == "Hyderabad") & (f.col("station_id") == "302"), "STN_HYD_02")
        .when((f.col("city") == "Mumbai")    & (f.col("station_id") == "401"), "STN_MUM_01")
        .when((f.col("city") == "Mumbai")    & (f.col("station_id") == "402"), "STN_MUM_02")
        .when((f.col("city") == "Delhi")     & (f.col("station_id") == "501"), "STN_DEL_01")
        .when((f.col("city") == "Delhi")     & (f.col("station_id") == "502"), "STN_DEL_02")
        .when((f.col("city") == "Pune")      & (f.col("station_id") == "601"), "STN_PNE_01")
        .when((f.col("city") == "Pune")      & (f.col("station_id") == "602"), "STN_PNE_02")
        .otherwise(f.col("station_id"))
    )
df.display()

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,rainfall,station_id,data_provider,uv_index,visibility,condition,feels_like_temperature,_rescued_data,ingestion_time,load_date,source_file,batch_id,record_status,_change_type,_commit_version,_commit_timestamp
Chennai,India,2026-03-01 00:00:00,36.4,84.0,18.7,1005.4,Foggy,7.3,STN_CHN_02,AtmosAPI,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Bangalore,India,2026-03-01 00:00:00,24.3,56.0,11.6,904.2,Partly Cloudy,0.9,STN_BLR_02,ClimaData,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Hyderabad,India,2026-03-01 00:00:00,29.7,68.0,7.2,1002.8,Rainy,5.7,STN_HYD_01,SkyWatch,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Mumbai,India,2026-03-01 00:00:00,30.9,84.0,19.6,1009.6,Foggy,9.5,STN_MUM_02,WeatherPro,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Delhi,India,2026-03-01 00:00:00,38.9,52.0,27.9,996.4,Partly Cloudy,3.2,STN_DEL_01,AtmosAPI,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Pune,India,2026-03-01 00:00:00,33.6,60.0,9.4,944.9,Partly Cloudy,0.8,STN_PNE_01,ClimaData,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Chennai,India,2026-03-01 01:00:00,36.0,64.0,17.6,1009.5,Clear,14.9,STN_CHN_02,WeatherPro,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Bangalore,India,2026-03-01 01:00:00,23.6,59.0,14.2,908.2,Partly Cloudy,9.2,STN_BLR_01,SkyWatch,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Hyderabad,India,2026-03-01 01:00:00,31.7,64.0,18.8,994.8,Cloudy,3.5,STN_HYD_01,ClimaData,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z
Mumbai,India,2026-03-01 01:00:00,27.3,86.0,13.9,1008.6,Haze,15.2,STN_MUM_01,WeatherPro,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z


In [0]:

station_master = spark.read.table("Weather_Analytics.bronze.bronze_station_master") 

df_joined = df.alias("w").join(
    station_master.alias("s"),
    F.col("w.station_id") == F.col("s.station_id"),
    "left"
).select(
    "w.*",
    F.col("s.station_id").alias("ref_station_id"),
    F.col("s.station_zone"),
    F.col("s.is_active"),
    F.col("s.commissioned")
)
df_joined.display()

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,rainfall,station_id,data_provider,uv_index,visibility,condition,feels_like_temperature,_rescued_data,ingestion_time,load_date,source_file,batch_id,record_status,_change_type,_commit_version,_commit_timestamp,ref_station_id,station_zone,is_active,commissioned
Chennai,India,2026-03-01 00:00:00,36.4,84.0,18.7,1005.4,Foggy,7.3,STN_CHN_02,AtmosAPI,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_CHN_02,South,true,2024-01-01
Bangalore,India,2026-03-01 00:00:00,24.3,56.0,11.6,904.2,Partly Cloudy,0.9,STN_BLR_02,ClimaData,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_BLR_02,South,true,2024-01-01
Hyderabad,India,2026-03-01 00:00:00,29.7,68.0,7.2,1002.8,Rainy,5.7,STN_HYD_01,SkyWatch,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_HYD_01,South,true,2024-01-01
Mumbai,India,2026-03-01 00:00:00,30.9,84.0,19.6,1009.6,Foggy,9.5,STN_MUM_02,WeatherPro,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_MUM_02,West,true,2024-01-01
Delhi,India,2026-03-01 00:00:00,38.9,52.0,27.9,996.4,Partly Cloudy,3.2,STN_DEL_01,AtmosAPI,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_DEL_01,North,true,2024-01-01
Pune,India,2026-03-01 00:00:00,33.6,60.0,9.4,944.9,Partly Cloudy,0.8,STN_PNE_01,ClimaData,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_PNE_01,West,true,2024-01-01
Chennai,India,2026-03-01 01:00:00,36.0,64.0,17.6,1009.5,Clear,14.9,STN_CHN_02,WeatherPro,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_CHN_02,South,true,2024-01-01
Bangalore,India,2026-03-01 01:00:00,23.6,59.0,14.2,908.2,Partly Cloudy,9.2,STN_BLR_01,SkyWatch,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_BLR_01,South,true,2024-01-01
Hyderabad,India,2026-03-01 01:00:00,31.7,64.0,18.8,994.8,Cloudy,3.5,STN_HYD_01,ClimaData,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_HYD_01,South,true,2024-01-01
Mumbai,India,2026-03-01 01:00:00,27.3,86.0,13.9,1008.6,Haze,15.2,STN_MUM_01,WeatherPro,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw,insert,2,2026-04-20T03:20:54.000Z,STN_MUM_01,West,true,2024-01-01


In [0]:
df_joined = df_joined.withColumn(
    "quarantine_reason",
    F.when(F.col("_rescued_data").isNotNull(), "corrupt_record")
     .when(F.col("event_time").isNull(), "invalid_timestamp")
     .when((F.col("temperature") > 60) | (F.col("temperature") < -20), "bad_temp")
     .when((F.col("humidity") > 100) | (F.col("humidity") < 0), "bad_humidity")
     .when(F.col("wind_speed") < 0, "bad_wind")
     .when(F.col("ref_station_id").isNull(), "bad_station")   
)

In [0]:
bad_df = df_joined.filter(F.col("quarantine_reason").isNotNull())
clean_df = df_joined.filter(F.col("quarantine_reason").isNull())

In [0]:
clean_df = df_joined \
    .withColumn("event_time", F.expr("try_cast(event_time as timestamp)")) \
    .withColumn("temperature", F.col("temperature").cast("double")) \
    .withColumn("humidity", F.col("humidity").cast("int")) \
    .withColumn("wind_speed", F.col("wind_speed").cast("double")) \
    .withColumn("pressure", F.col("pressure").cast("double")) \
    .withColumn("rainfall", F.col("rainfall").cast("double"))

clean_df = clean_df \
    .withColumn("event_time", F.to_timestamp("event_time")) \
    .withColumn("event_date", F.to_date("event_time")) \
    .withColumn("event_hour", F.hour("event_time")) \
    .withColumn(
        "temperature_band",
        F.when(F.col("temperature") < 15, "Cold")
         .when(F.col("temperature") < 30, "Normal")
         .when(F.col("temperature") <= 40, "Hot")
         .otherwise("Extreme")
    ) \
    .withColumn(
        "weather_severity_score",
        F.when(F.col("temperature") > 40, 3).otherwise(0) +
        F.when(F.col("rainfall") > 30, 2).otherwise(0) +
        F.when(F.col("wind_speed") > 40, 2).otherwise(0)
    )

In [0]:
clean_df.count()

1620

In [0]:
df_joined.select("station_id", "ref_station_id").display()

station_id,ref_station_id
STN_CHN_02,STN_CHN_02
STN_BLR_02,STN_BLR_02
STN_HYD_01,STN_HYD_01
STN_MUM_02,STN_MUM_02
STN_DEL_01,STN_DEL_01
STN_PNE_01,STN_PNE_01
STN_CHN_02,STN_CHN_02
STN_BLR_01,STN_BLR_01
STN_HYD_01,STN_HYD_01
STN_MUM_01,STN_MUM_01


In [0]:

window_fn = Window.partitionBy("city", "event_time") \
    .orderBy(F.col("ingestion_time").desc())

clean_df = clean_df.withColumn(
    "row_num",
    F.row_number().over(window_fn)
).filter("row_num = 1").drop("row_num")

bad_df = bad_df.drop("ref_station_id")
clean_df = clean_df.drop("ref_station_id")


In [0]:
clean_df.count()


1417

In [0]:
bad_df.count()

54

In [0]:
bad_df.groupBy("quarantine_reason").count().display()

quarantine_reason,count
bad_station,19
bad_temp,24
bad_humidity,11


In [0]:
clean_df = clean_df.drop(
    "_change_type",
    "_commit_version",
    "_commit_timestamp"
)
bad_df = bad_df.drop(
    "_change_type",
    "_commit_version",
    "_commit_timestamp"
)

In [0]:

bad_df.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .format("delta") \
    .saveAsTable("Weather_Analytics.silver.silver_weather_quarantine")

In [0]:

if not spark.catalog.tableExists("Weather_Analytics.silver.silver_weather_clean"):
    clean_df.limit(0).write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable("Weather_Analytics.silver.silver_weather_clean")

In [0]:
spark.read.table("Weather_Analytics.silver.silver_weather_clean").printSchema()

root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: integer (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- rainfall: double (nullable = true)
 |-- station_id: string (nullable = true)
 |-- data_provider: string (nullable = true)
 |-- uv_index: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- condition: string (nullable = true)
 |-- feels_like_temperature: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- load_date: date (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- record_status: string (nullable = true)
 |-- station_zone: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- comm

In [0]:
clean_df.createOrReplaceTempView("updates")

spark.sql("""
MERGE INTO Weather_Analytics.silver.silver_weather_clean t
USING updates s
ON t.city = s.city AND t.event_time = s.event_time

WHEN MATCHED AND s.ingestion_time > t.ingestion_time THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:

latest_version = spark.sql("""
DESCRIBE HISTORY Weather_Analytics.bronze.bronze_weather
""").selectExpr("max(version)").collect()[0][0]

spark.sql(f"""
INSERT INTO Weather_Analytics.silver.pipeline_control
VALUES ('bronze_weather', {latest_version}, current_timestamp())
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
final=spark.read.table("weather_analytics.silver.silver_weather_clean")
final.display()

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,rainfall,station_id,data_provider,uv_index,visibility,condition,feels_like_temperature,_rescued_data,ingestion_time,load_date,source_file,batch_id,record_status,station_zone,is_active,commissioned,quarantine_reason,event_date,event_hour,temperature_band,weather_severity_score
Bangalore,India,null,31.7,74,14.1,null,Thunderstorm,6.4,STN_BLR_01,SkyWatch,null,null,null,null,null,2026-04-20T03:21:09.265Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_08.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,null,null,Hot,0
Bangalore,India,2026-02-22T00:00:00.000Z,27.5,57,11.9,913.3,Foggy,1.5,STN_BLR_02,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,0,Normal,0
Bangalore,India,2026-02-22T01:00:00.000Z,31.1,64,5.9,913.8,Partly Cloudy,3.2,STN_BLR_01,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,1,Hot,0
Bangalore,India,2026-02-22T02:00:00.000Z,29.8,60,12.1,911.9,Clear,1.2,STN_BLR_02,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,2,Normal,0
Bangalore,India,2026-02-22T03:00:00.000Z,30.9,50,7.2,902.6,Clear,6.3,STN_BLR_01,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,3,Hot,0
Bangalore,India,2026-02-22T04:00:00.000Z,28.0,57,7.0,900.7,Haze,9.1,STN_BLR_01,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,4,Normal,0
Bangalore,India,2026-02-22T05:00:00.000Z,24.9,50,5.5,903.0,Partly Cloudy,8.0,STN_BLR_01,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,5,Normal,0
Bangalore,India,2026-02-22T06:00:00.000Z,27.8,71,16.5,901.0,Foggy,4.3,STN_BLR_01,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,6,Normal,0
Bangalore,India,2026-02-22T07:00:00.000Z,24.2,55,14.3,905.5,Thunderstorm,4.7,STN_BLR_02,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,7,Normal,0
Bangalore,India,2026-02-22T08:00:00.000Z,26.2,61,17.8,914.5,Clear,8.2,STN_BLR_01,WeatherPro,null,null,null,null,null,2026-04-20T03:21:14.396Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_10.csv,2026-04-13 04:49:34,raw,South,true,2024-01-01,null,2026-02-22,8,Normal,0


In [0]:
import pyspark.sql.functions as F

silver_table = "Weather_Analytics.silver.silver_weather_clean"
audit_table  = "Weather_Analytics.silver.weather_audit_tbl"
# 1: GET LATEST MERGE VERSION

history_df = spark.sql(f"DESCRIBE HISTORY {silver_table}")

latest = (history_df
          .filter("operation = 'MERGE'")
          .orderBy("version", ascending=False)
          .limit(1))

if latest.count() == 0:
    print("No MERGE found. Skipping audit.")

else:
    cur_ver = latest.first()["version"]
    prev_ver = cur_ver - 1

    if prev_ver < 0:
        print("No previous version to compare.")

    else:
        print(f"Comparing version {prev_ver} → {cur_ver}")

 # 2: READ REQUIRED COLUMNS ONLY
    
        prev_df = spark.read.format("delta") \
            .option("versionAsOf", prev_ver) \
            .table(silver_table) \
            .select("city", "event_time", "temperature", "humidity", "wind_speed", "ingestion_time")

        curr_df = spark.read.format("delta") \
            .option("versionAsOf", cur_ver) \
            .table(silver_table) \
            .select("city", "event_time", "temperature", "humidity", "wind_speed", "ingestion_time")

        prev_df = prev_df.withColumn("event_time", F.col("event_time").cast("string"))
        curr_df = curr_df.withColumn("event_time", F.col("event_time").cast("string"))

#  3: JOIN
       
        df = prev_df.alias("p").join(
            curr_df.alias("c"),
            on=["city", "event_time"],
            how="full"
        )
 # 4: CHANGE DETECTION
       
        changes_df = df.filter(
            (F.col("p.temperature") != F.col("c.temperature")) |
            (F.col("p.humidity") != F.col("c.humidity")) |
            (F.col("p.wind_speed") != F.col("c.wind_speed")) |
            (F.col("p.ingestion_time") != F.col("c.ingestion_time")) |
            F.col("p.city").isNull() |
            F.col("c.city").isNull()
        ).select(
            F.coalesce(F.col("c.city"), F.col("p.city")).alias("city"),
            F.coalesce(F.col("c.event_time"), F.col("p.event_time")).alias("event_time"),
            F.when(F.col("p.city").isNull(), "insert")
             .when(F.col("c.city").isNull(), "delete")
             .otherwise("update")
             .alias("change_type"),
            F.current_timestamp().alias("audit_time")
        )
        
        changes_df.display()

        changes_df.write \
            .mode("append") \
            .format("delta") \
            .option("mergeSchema", "true") \
            .saveAsTable(audit_table)

        print(" Audit completed successfully")

Comparing version 0 → 1


city,event_time,change_type,audit_time
Bangalore,null,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 00:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 01:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 02:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 03:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 04:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 05:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 06:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 07:00:00,insert,2026-04-20T03:23:36.599Z
Bangalore,2026-02-22 08:00:00,insert,2026-04-20T03:23:36.599Z


 Audit completed successfully
